<a href="https://colab.research.google.com/github/anushkapatel110-ui/Careloop-clinics-no-show-guard-PROJECT/blob/main/03_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#3.1
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("appointments_clean.csv")   # do NOT sort or shuffle after this

X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent",
        "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X columns:", X.shape[1])
print("train:", len(X_train), "test:", len(X_test))
print("no-shows in test:", y_test.sum())
print("baseline accuracy:", round((y_test == 0).mean(), 4), f"({(y_test == 0).sum()} of {len(y_test)})")

X columns: 11
train: 792 test: 198
no-shows in test: 49
baseline accuracy: 0.7525 (149 of 198)


In [ ]:
#3.2: logistic regression on scaled features
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on training only
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

pred = lr.predict(X_test_s)
prob = lr.predict_proba(X_test_s)[:, 1]

print("accuracy :", round(accuracy_score(y_test, pred), 3))
print("confusion matrix:\n", confusion_matrix(y_test, pred))
print("precision:", round(precision_score(y_test, pred), 3))
print("recall   :", round(recall_score(y_test, pred), 3))
print("F1       :", round(f1_score(y_test, pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, prob), 3))

coefs = pd.Series(lr.coef_[0], index=X.columns)
print("\ncoefficients:\n", coefs.round(3))

i = list(X.columns).index("reminder_sent")
coef_r = lr.coef_[0][i]
scale_r = scaler.scale_[i]
print("\nreminder_sent coef (scaled):", round(coef_r, 3), "| scale (SD):", round(scale_r, 3))
print("exp(coef) scaled          :", round(np.exp(coef_r), 2))
print("exp(coef / scale) No->Yes :", round(np.exp(coef_r / scale_r), 2))

accuracy : 0.788
confusion matrix:
 [[139  10]
 [ 32  17]]
precision: 0.63
recall   : 0.347
F1       : 0.447
ROC-AUC  : 0.789

coefficients:
 age                       -0.254
distance_km                0.253
fee_inr                    0.453
lead_days                  0.614
prior_no_shows             0.650
reminder_sent             -0.581
booking_channel_App       -0.047
booking_channel_Phone      0.216
booking_channel_Walk-In   -0.186
visit_type_Consultation    0.016
visit_type_Follow-up      -0.016
dtype: float64

reminder_sent coef (scaled): -0.581 | scale (SD): 0.495
exp(coef) scaled          : 0.56
exp(coef / scale) No->Yes : 0.31


exp(coef) on the scaled feature is 0.56, but the real No→Yes effect is exp(coef/scale) = 0.31. They differ because 0.56 is for a one-SD change (0.495 units) while No→Yes is a full unit, about 2 SDs. In this model, holding the other features constant, the odds of a no-show are about 69% lower with a reminder; this is an association in observational data, not proof of cause.

In [ ]:
#3.3: decision tree and overfitting
from sklearn.tree import DecisionTreeClassifier

tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_d3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

for name, t in [("full depth", tree_full), ("max_depth=3", tree_d3)]:
    print(name,
          "| train acc:", round(t.score(X_train, y_train), 3),
          "| test acc:", round(t.score(X_test, y_test), 3))
print("full tree depth:", tree_full.get_depth())

full depth | train acc: 1.0 | test acc: 0.672
max_depth=3 | train acc: 0.822 | test acc: 0.778
full tree depth: 15


The fully grown decision tree achieves perfect training accuracy (1.00) but performs poorly on the test set (0.672), falling below the 0.7525 baseline of always predicting “Attended.” This indicates overfitting, as the model has memorised the training data rather than learning meaningful patterns. In contrast, the depth-3 tree shows a smaller performance gap (0.822 training accuracy vs. 0.778 test accuracy), demonstrating better generalisation to unseen data.

In [ ]:
#Task 3.4
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("RF train acc:", round(rf.score(X_train, y_train), 3))
print("RF test acc :", round(accuracy_score(y_test, rf_pred), 3))
print("RF recall   :", round(recall_score(y_test, rf_pred), 3))
print("RF ROC-AUC  :", round(roc_auc_score(y_test, rf_prob), 3))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\ntop 3 importances:\n", imp.head(3).round(3))

# comparison table (reporting only)
d3_pred = tree_d3.predict(X_test)
d3_prob = tree_d3.predict_proba(X_test)[:, 1]
comparison = pd.DataFrame({
    "accuracy": [accuracy_score(y_test, pred), accuracy_score(y_test, d3_pred), accuracy_score(y_test, rf_pred)],
    "recall":   [recall_score(y_test, pred),   recall_score(y_test, d3_pred),   recall_score(y_test, rf_pred)],
    "ROC-AUC":  [roc_auc_score(y_test, prob),  roc_auc_score(y_test, d3_prob),  roc_auc_score(y_test, rf_prob)],
}, index=["Logistic regression", "Decision tree (depth 3)", "Random forest"]).round(3)
print("\n", comparison)

RF train acc: 1.0
RF test acc : 0.747
RF recall   : 0.306
RF ROC-AUC  : 0.706

top 3 importances:
 lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64

                          accuracy  recall  ROC-AUC
Logistic regression         0.788   0.347    0.789
Decision tree (depth 3)     0.778   0.224    0.700
Random forest               0.747   0.306    0.706


Logistic regression demonstrates the strongest overall performance, achieving the highest accuracy (0.788), recall (0.347), and AUC (0.789). It is also the most interpretable model, making its results easier to explain. However, this table is intended for reporting purposes only; the final model selection is formally made in Section 3.5, using training data exclusively.

In [ ]:
#Task 3.5
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# (a) tune the forest on the training set
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print("best_score_ (CV ROC-AUC):", round(grid.best_score_, 3))

# (b) choose the model by 5-fold CV ROC-AUC on the training set
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv = {
    "Logistic regression": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                                               X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                     X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned forest": grid.best_score_,
}
for k, v in cv.items():
    print(f"{k:25s} {v:.3f}")
winner = max(cv, key=cv.get)
print("winner:", winner)

# (c) threshold from out-of-fold training probabilities (winner = logistic regression expected)
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
for t in [0.5, 0.35]:
    f = (oof >= t).astype(int)
    print(f"threshold {t}: precision {precision_score(y_train, f):.3f} | "
          f"recall {recall_score(y_train, f):.3f} | F1 {f1_score(y_train, f):.3f} | flagged {f.sum()}")

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (CV ROC-AUC): 0.767
Logistic regression       0.774
Decision tree (depth 3)   0.678
Random forest             0.747
Tuned forest              0.767
winner: Logistic regression
threshold 0.5: precision 0.575 | recall 0.249 | F1 0.347 | flagged 73
threshold 0.35: precision 0.513 | recall 0.467 | F1 0.489 | flagged 154


- Logistic regression achieves the highest cross-validated AUC (0.774), based solely on the training data. The test set was not used during model selection, ensuring an unbiased evaluation of the model's performance on unseen data.

- Recall is particularly important because failing to identify a patient who will miss their appointment can result in an unused doctor’s slot, whereas making an additional reminder call requires only about a minute.

- In a real-world setting, clinics would choose the classification threshold based on the front desk's capacity to make follow-up calls each evening and the potential cost of unfilled appointment slots.

In [ ]:
THRESH = 0.35

lr_pipe.fit(X_train, y_train)                     # pipeline fits its own scaler
test_prob = lr_pipe.predict_proba(X_test)[:, 1]
test_flag = (test_prob >= THRESH).astype(int)

print("precision:", round(precision_score(y_test, test_flag), 3))
print("recall   :", round(recall_score(y_test, test_flag), 3))
print("flagged  :", test_flag.sum())
print("ROC-AUC  :", round(roc_auc_score(y_test, test_prob), 3))

out = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_prob.round(4),
    "flag": test_flag,
})
out.to_csv("test_risk_scores.csv", index=False)
print(out.shape, "| rows with flag=1:", out["flag"].sum())

precision: 0.571
recall   : 0.571
flagged  : 49
ROC-AUC  : 0.789
(198, 4) | rows with flag=1: 49
